# Content-Based Book Recommendation System for Company U
## Using Genres and TF-IDF Similarity

This notebook implements a complete content-based filtering system that:
- Parses user book preferences from CSV data
- Merges books with their genre classifications
- Creates TF-IDF vectors from book features (titles, genres, metadata)
- Builds similarity matrices using cosine similarity
- Provides recommendations based on books a user has already read
- **Testable on ANY user in the dataset**

### System Architecture
1. **Data Loading**: Parse user-book relationships from CSV
2. **Feature Engineering**: Extract genres from classifier output
3. **Vectorization**: TF-IDF transformation of book content
4. **Similarity**: Cosine similarity between books
5. **Recommendation**: Content-based recall using similarity scores

## Section 1: Load and Parse the Book Data

In [45]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Load the user-book data
user_books_df = pd.read_csv('company_u.csv', header=0)
# Rename columns to match expected names
user_books_df.columns = ['User', 'Books']
print("Loaded user data:")
print(f"Total users: {len(user_books_df)}")
print(f"\nFirst 3 users:")
print(user_books_df.head(3))
print(f"\nUser IDs range: {user_books_df['User'].min()} to {user_books_df['User'].max()}")

Loaded user data:
Total users: 298

First 3 users:
   User                       Books
0 User002 A guide to the project management body of know...
1 User003    Ideology : an introduction / Terry Eagleton.
2 User004 How to think on your feet : a revolutionary te...

User IDs range: User002 to User300


In [ ]:
# Parse books: split by " / " (new book delimiter) and " . " (continuation)
def parse_books(books_string):
  """
  Parse books from a user's book string.
  Books are delimited by " / " or " . "
  Extract title (before " / by " or " ; ")
  """
  if pd.isna(books_string):
    return []
  
  # Replace period-space-comma with period (handle edge cases)
  books_str = str(books_string)
  
  # Split by " / " (primary delimiter for new books)
  books = books_str.split(' / ')
  
  titles = []
  for book in books:
    # Remove the author part (after " by " or " ; ")
    if ' by ' in book:
      title = book.split(' by ')[0].strip()
    elif ' ; ' in book:
      title = book.split(' ; ')[0].strip()
    else:
      title = book.strip()
    
    # Remove any trailing dots
    title = title.rstrip('.')
    
    if title: # Only add non-empty titles
      titles.append(title)
  
  return titles

# Apply parsing to create user-book pairs
user_book_pairs = []
for _, row in user_books_df.iterrows():
  user = row['User']
  books = parse_books(row['Books'])
  for book in books:
    user_book_pairs.append({'User': user, 'Title': book})

user_books_parsed = pd.DataFrame(user_book_pairs)
print(f"\nTotal user-book pairs: {len(user_books_parsed)}")
print(f"Unique books: {user_books_parsed['Title'].nunique()}")
print(f"\nFirst 5 parsed pairs:")
print(user_books_parsed.head())
print(f"\nBooks for User006:")
print(user_books_parsed[user_books_parsed['User'] == 'User006'])


Total user-book pairs: 1000
Unique books: 910

First 5 parsed pairs:
   User                       Title
0    id                       books
1 User002 A guide to the project management body of know...
2 User002            Project Management Institute
3 User003             Ideology : an introduction
4 User003                   Terry Eagleton

Books for User180:
    User                       Title
612 User180 The gospels of Queen Keran : transition to pos...
613 User180             Emma Chookaszian ; preface
614 User180                  Emma Ch'ugaszyan


In [46]:
# Load genre classifier output
genres_df = pd.read_csv('company_u_genres_output.csv')
print("Loaded genre data:")
print(f"Total classified books: {len(genres_df)}")
print(f"\nColumns: {genres_df.columns.tolist()}")
print(f"\nFirst 5 classified books:")
print(genres_df.head())
print(f"\nGenre distribution:")
print(genres_df['Genres'].value_counts().head(10))

Loaded genre data:
Total classified books: 668

Columns: ['Title', 'Number', 'language', 'Genres']

First 5 classified books:
                        Title Number \
0              the bastard of istanbul   3.0  
1 business model generation a handbook for visio...   3.0  
2                  college algebra   3.0  
3                 kafka on the shore   3.0  
4    the norton anthology of american literature   3.0  

            language                   Genres 
0            English         Horror, Fantasy, Thriller 
1            English        Nonfiction, Fantasy, Mystery 
2 Armenian (Latin romanization) Historical Fiction, Nonfiction, Biography 
3            English       Nonfiction, Fantasy, Thriller 
4            English         Thriller, Fantasy, Mystery 

Genre distribution:
Genres
Historical Fiction, Nonfiction, Biography  90
Nonfiction, Mystery, Fantasy         78
Nonfiction, Mystery, Young Adult       65
Nonfiction                  58
Nonfiction, Young Adult, Mystery       50
No

In [47]:
# Merge user-book pairs with genres (case-insensitive matching)
# Normalize titles for matching
genres_df['Title_lower'] = genres_df['Title'].str.lower().str.strip()
user_books_parsed['Title_lower'] = user_books_parsed['Title'].str.lower().str.strip()

# Merge on normalized titles
user_books_with_genres = user_books_parsed.merge(
  genres_df[['Title_lower', 'Genres', 'language']],
  left_on='Title_lower',
  right_on='Title_lower',
  how='left'
)

# Check for unmatched books
unmatched = user_books_with_genres[user_books_with_genres['Genres'].isna()]
print(f"Matched books: {len(user_books_with_genres) - len(unmatched)}")
print(f"Unmatched books: {len(unmatched)}")

if len(unmatched) > 0:
  print(f"\nSample unmatched books:")
  print(unmatched[['User', 'Title']].drop_duplicates().head(10))

# Remove unmatched books (we only work with classified books)
user_books_with_genres = user_books_with_genres.dropna(subset=['Genres'])

print(f"\nFinal user-book-genre dataset:")
print(f"Total pairs: {len(user_books_with_genres)}")
print(f"Unique books: {user_books_with_genres['Title'].nunique()}")
print(f"Unique users: {user_books_with_genres['User'].nunique()}")
print(f"\nSample data:")
print(user_books_with_genres.head(10))

Matched books: 121
Unmatched books: 879

Sample unmatched books:
    User                       Title
0    id                       books
2  User002            Project Management Institute
3  User003             Ideology : an introduction
4  User003                   Terry Eagleton
5  User004 How to think on your feet : a revolutionary te...
6  User004 by Mark Bergren, Molly Cox, Jim Detmar ; illus...
8  User006 by Martin Daly and Margo Wilson., Everything y...
9  User006 by Patricia Noble Sullivan, Grace Yi Qiu Zhong...
10 User006 by Armand Marie Leroi., The clash of civilizat...
11 User006 by Samuel P. Huntington., ESL grammar handbook...

Final user-book-genre dataset:
Total pairs: 121
Unique books: 116
Unique users: 121

Sample data:
    User                       Title \
1  User002 A guide to the project management body of know...  
7  User006                      Homicide  
23 User009                      Calculus  
25 User010              Essentials of marketing  
31 User012    

## Section 2: Extract Genre Features from Book Titles and Metadata

In [33]:
# Create a unique books dataframe with combined features
books_unique = user_books_with_genres[['Title', 'Genres', 'language']].drop_duplicates()
print(f"Unique books for vectorization: {len(books_unique)}")

# Create feature "soup": combine title words + genres + language
# Genres are already comma-separated, so we replace commas with spaces for tokenization
def create_feature_soup(row):
  """
  Create a feature soup combining:
  - Book title (multiple times for emphasis)
  - Genres (comma-separated list)
  - Language (for multilingual books)
  """
  title = str(row['Title']).lower()
  genres = str(row['Genres']).lower()
  language = str(row['language']).lower() if pd.notna(row['language']) else ''
  
  # Repeat genres for emphasis (genres are more important than title words)
  # Genre words carry more semantic meaning
  soup = f"{title} {genres} {genres} {language}"
  
  return soup

books_unique['soup'] = books_unique.apply(create_feature_soup, axis=1)

print(f"\nSample soups created:")
for idx, row in books_unique.head(5).iterrows():
  print(f"\n{row['Title']}:")
  print(f" Genres: {row['Genres']}")
  print(f" Soup: {row['soup'][:100]}...")

Unique books for vectorization: 116

Sample soups created:

A guide to the project management body of knowledge:
 Genres: Nonfiction
 Soup: a guide to the project management body of knowledge nonfiction nonfiction english...

Homicide:
 Genres: Mystery
 Soup: homicide mystery mystery spanish...

Calculus:
 Genres: Nonfiction, Thriller, Mystery
 Soup: calculus nonfiction, thriller, mystery nonfiction, thriller, mystery ro...

Essentials of marketing:
 Genres: Nonfiction, Mystery, Thriller
 Soup: essentials of marketing nonfiction, mystery, thriller nonfiction, mystery, thriller english...

Calculus with analytic geometry:
 Genres: Nonfiction, Mystery, Fantasy
 Soup: calculus with analytic geometry nonfiction, mystery, fantasy nonfiction, mystery, fantasy english...


## Section 3: Create TF-IDF Vectors for Book Content

In [48]:
# Create TF-IDF vectors
tfidf = TfidfVectorizer(
  max_features=500, # Limit features to top 500 most important terms
  min_df=1, # Include all terms (even if appearing in only one book)
  max_df=0.95, # Exclude terms appearing in >95% of documents
  ngram_range=(1, 2), # Use unigrams and bigrams
  sublinear_tf=True # Apply sublinear term frequency scaling
)

# Fit and transform the soup
tfidf_matrix = tfidf.fit_transform(books_unique['soup'])

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f" - Books: {tfidf_matrix.shape[0]}")
print(f" - Features: {tfidf_matrix.shape[1]}")

# Get feature names (for interpretation)
feature_names = tfidf.get_feature_names_out()
print(f"\nTop 20 features by frequency:")
print(feature_names[:20])

# Show which features are important for a sample book
sample_idx = 0
sample_title = books_unique.iloc[sample_idx]['Title']
sample_soup = books_unique.iloc[sample_idx]['soup']
sample_vector = tfidf_matrix[sample_idx].toarray().flatten()
top_indices = sample_vector.argsort()[-10:][::-1]

print(f"\nTop 10 TF-IDF features for '{sample_title}':")
for idx in top_indices:
  if sample_vector[idx] > 0:
    print(f" {feature_names[idx]}: {sample_vector[idx]:.4f}")

TF-IDF Matrix Shape: (116, 500)
 - Books: 116
 - Features: 500

Top 20 features by frequency:
['48' 'adult' 'adult english' 'adult mystery' 'adult nonfiction'
 'adult science' 'adult thriller' 'after' 'analysis' 'and' 'and engineers'
 'and its' 'and the' 'applications' 'applications nonfiction' 'armenia'
 'armenian' 'armenian latin' 'biography' 'biography armenian']

Top 10 TF-IDF features for 'A guide to the project management body of knowledge':
 project: 0.3426
 project management: 0.3426
 of knowledge: 0.3426
 the project: 0.3426
 management: 0.3152
 to the: 0.2957
 guide to: 0.2957
 guide: 0.2806
 to: 0.2161
 nonfiction nonfiction: 0.1904


## Section 4: Build Similarity Matrix

In [49]:
# Compute cosine similarity matrix between all books
similarity_matrix = cosine_similarity(tfidf_matrix)

print(f"Similarity Matrix Shape: {similarity_matrix.shape}")
print(f" - Square matrix for {similarity_matrix.shape[0]} books")

# Create a mapping from title to book index
title_to_idx = {title: idx for idx, title in enumerate(books_unique['Title'])}

# Analyze similarity distribution
import numpy as np
similarity_scores = similarity_matrix[np.triu_indices_from(similarity_matrix, k=1)]
print(f"\nSimilarity Score Statistics:")
print(f" Mean: {similarity_scores.mean():.4f}")
print(f" Std: {similarity_scores.std():.4f}")
print(f" Min: {similarity_scores.min():.4f}")
print(f" Max: {similarity_scores.max():.4f}")
print(f" 25%: {np.percentile(similarity_scores, 25):.4f}")
print(f" 50%: {np.percentile(similarity_scores, 50):.4f}")
print(f" 75%: {np.percentile(similarity_scores, 75):.4f}")

# Show top 5 most similar books to a sample book
sample_idx = 1
sample_title = books_unique.iloc[sample_idx]['Title']
sample_genres = books_unique.iloc[sample_idx]['Genres']
similarities = similarity_matrix[sample_idx]
top_similar_idx = similarities.argsort()[-6:-1][::-1] # Top 5 (excluding itself)

print(f"\n\nTop 5 Most Similar Books to '{sample_title}'")
print(f"(Genres: {sample_genres})")
print("-" * 80)
for rank, idx in enumerate(top_similar_idx, 1):
  similar_title = books_unique.iloc[idx]['Title']
  similar_genres = books_unique.iloc[idx]['Genres']
  sim_score = similarities[idx]
  print(f"{rank}. [{sim_score:.4f}] {similar_title}")
  print(f"  Genres: {similar_genres}\n")

Similarity Matrix Shape: (116, 116)
 - Square matrix for 116 books

Similarity Score Statistics:
 Mean: 0.0942
 Std: 0.1523
 Min: 0.0000
 Max: 1.0000
 25%: 0.0161
 50%: 0.0337
 75%: 0.1043


Top 5 Most Similar Books to 'Homicide'
(Genres: Mystery)
--------------------------------------------------------------------------------
1. [0.4696] Renewable energy
  Genres: Nonfiction, Fantasy, Mystery

2. [0.3455] Spanish for mastery 2
  Genres: Thriller, Nonfiction, Mystery

3. [0.1288] The alchemist
  Genres: Fantasy, Nonfiction, Mystery

4. [0.1242] The namesake
  Genres: Thriller, Nonfiction, Mystery

5. [0.1227] Essentials of marketing
  Genres: Nonfiction, Mystery, Thriller



## Section 5: Implement Content-Based Recommendation Function

In [50]:
def get_recommendations(user_id, num_recommendations=10, min_similarity=0.0):
  """
  Get book recommendations for a specific user based on content similarity.
  
  Algorithm:
  1. Get all books the user has read
  2. For each book, find similar books using similarity matrix
  3. Aggregate similarity scores for books user hasn't read
  4. Return top N recommendations
  
  Args:
    user_id (str): User ID (e.g., 'User162')
    num_recommendations (int): Number of recommendations to return (default: 10)
    min_similarity (float): Minimum similarity threshold (default: 0.0)
  
  Returns:
    dict: Contains user info, read books, recommendations dataframe
  """
  
  # Find books this user has read
  user_books = user_books_with_genres[user_books_with_genres['User'] == user_id]['Title'].unique()
  
  if len(user_books) == 0:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'{user_id} not found in dataset'
    }
  
  # Get indices of books user has read
  read_indices = []
  for book in user_books:
    if book in title_to_idx:
      read_indices.append(title_to_idx[book])
  
  if len(read_indices) == 0:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': f'No classified books found for {user_id}'
    }
  
  # Aggregate similarities: for each unread book, compute average similarity to read books
  book_scores = {}
  for book_idx in range(len(books_unique)):
    if book_idx not in read_indices:
      # Compute average similarity to books user has read
      similarities = [similarity_matrix[book_idx][read_idx] for read_idx in read_indices]
      avg_similarity = np.mean(similarities)
      max_similarity = np.max(similarities)
      
      # Use weighted average of max and mean (emphasize highest similarities)
      score = 0.6 * max_similarity + 0.4 * avg_similarity
      
      if score >= min_similarity:
        book_scores[book_idx] = {
          'title': books_unique.iloc[book_idx]['Title'],
          'genres': books_unique.iloc[book_idx]['Genres'],
          'language': books_unique.iloc[book_idx]['language'],
          'avg_similarity': avg_similarity,
          'max_similarity': max_similarity,
          'score': score
        }
  
  # Sort by score and get top N
  if not book_scores:
    return {
      'user_id': user_id,
      'status': 'error',
      'message': 'No recommendations could be generated'
    }
  
  sorted_recs = sorted(book_scores.items(), key=lambda x: x[1]['score'], reverse=True)
  top_recs = sorted_recs[:num_recommendations]
  
  # Create recommendations dataframe
  recs_data = []
  for rank, (book_idx, scores) in enumerate(top_recs, 1):
    recs_data.append({
      'Rank': rank,
      'Title': scores['title'],
      'Genres': scores['genres'],
      'Language': scores['language'],
      'Similarity Score': scores['score'],
      'Max Similarity': scores['max_similarity'],
      'Avg Similarity': scores['avg_similarity']
    })
  
  recs_df = pd.DataFrame(recs_data)
  
  return {
    'user_id': user_id,
    'status': 'success',
    'num_books_read': len(user_books),
    'num_classified': len(read_indices),
    'books_read': list(user_books)[:5], # Show first 5
    'recommendations': recs_df
  }


# Test the function on a sample user
result = get_recommendations('User180', num_recommendations=10)
print(f"Recommendations for {result['user_id']}")
print(f"Status: {result['status']}")

if result['status'] == 'error':
  print(f"Error: {result['message']}")
else:
  print(f"Books read by user: {result['num_books_read']}")
  print(f"Classified books: {result['num_classified']}")
  print(f"\nSample books user has read:")
  for book in result['books_read']:
    print(f" - {book}")
  print(f"\nTop {len(result['recommendations'])} Recommendations:")
  print(result['recommendations'].to_string(index=False))

Recommendations for User180
Status: error
Error: User180 not found in dataset


## Section 6: Test Recommendations on Multiple Users

### Test ANY User - Interactive Testing Below

In [ ]:
# Get list of all users in the dataset
all_users = sorted(user_books_with_genres['User'].unique())
print(f"Total users in dataset: {len(all_users)}")
print(f"User ID range: {all_users[0]} to {all_users[-1]}")
print(f"Available users: {all_users}")

# Function to display recommendations nicely
def display_recommendations(user_id, num_recs=10):
  """Display recommendations for a user in a formatted way"""
  result = get_recommendations(user_id, num_recommendations=num_recs)
  
  if result['status'] == 'error':
    print(f"Error: {result['message']}")
    return
  
  print(f"\n{'='*90}")
  print(f"RECOMMENDATIONS FOR {result['user_id']}")
  print(f"{'='*90}")
  print(f"Books user has read: {result['num_books_read']}")
  print(f"Books classified: {result['num_classified']}")
  print(f"\nSample of books user likes:")
  for book in result['books_read']:
    genres_info = user_books_with_genres[user_books_with_genres['Title'] == book]['Genres'].iloc[0]
    print(f" {book}" )
    print(f"  → Genres: {genres_info}")
  
  print(f"\n{'-'*90}")
  print(f"TOP {len(result['recommendations'])} RECOMMENDATIONS:")
  print(f"{'-'*90}")
  
  for _, row in result['recommendations'].iterrows():
    print(f"#{int(row['Rank']):2d}. {row['Title']}")
    print(f"   Genres: {row['Genres']}")
    print(f"   Similarity Score: {row['Similarity Score']:.4f}" )
    print(f"   (Max: {row['Max Similarity']:.4f}, Avg: {row['Avg Similarity']:.4f})")
    print()

Total users in dataset: 121
User ID range: User002 to User299
Available users: ['User002', 'User006', 'User009', 'User010', 'User012', 'User013', 'User015', 'User017', 'User019', 'User021', 'User022', 'User023', 'User024', 'User027', 'User028', 'User029', 'User035', 'User037', 'User038', 'User039', 'User041', 'User046', 'User048', 'User053', 'User054', 'User062', 'User063', 'User064', 'User067', 'User069', 'User070', 'User071', 'User076', 'User077', 'User082', 'User083', 'User084', 'User085', 'User086', 'User093', 'User094', 'User096', 'User098', 'User100', 'User111', 'User112', 'User113', 'User114', 'User116', 'User120', 'User127', 'User134', 'User137', 'User140', 'User142', 'User145', 'User148', 'User151', 'User161', 'User164', 'User169', 'User170', 'User171', 'User172', 'User175', 'User177', 'User178', 'User179', 'User182', 'User184', 'User187', 'User190', 'User191', 'User192', 'User194', 'User196', 'User198', 'User199', 'User207', 'User208', 'User209', 'User211', 'User212', 'User21

In [ ]:
# INTERACTIVE TESTING - INPUT USER ID VIA DIALOG
# ===============================================
# This cell uses interactive widgets for easy user selection

try:
  from ipywidgets import Dropdown, IntSlider, Button, Output, VBox
  from IPython.display import display, clear_output
  
  # Create widgets
  user_dropdown = Dropdown(
    options=all_users,
    value=all_users[0],
    description='Select User:',
    style={'description_width': '120px'}
  )
  
  num_recs_slider = IntSlider(
    value=10,
    min=1,
    max=30,
    step=1,
    description='# Recommendations:',
    style={'description_width': '160px'}
  )
  
  execute_button = Button(
    description='Show Recommendations',
    button_style='info',
    tooltip='Click to get recommendations for selected user',
    icon='search'
  )
  
  output = Output()
  
  # Define callback function
  def on_button_click(b):
    with output:
      clear_output(wait=True)
      selected_user = user_dropdown.value
      num_recs = num_recs_slider.value
      display_recommendations(selected_user, num_recs=num_recs)
  
  execute_button.on_click(on_button_click)
  
  # Create layout
  controls = VBox([
    user_dropdown,
    num_recs_slider,
    execute_button
  ])
  
  display(controls)
  display(output)
  
  # Auto-run on first load
  on_button_click(None)
  
except ImportError:
  # Fallback if ipywidgets not available
  print("\nipywidgets not available. Using manual input instead.\n")
  print(f"Available users: {all_users}\n")
  
  user_input = input(f"Enter user ID (e.g., {all_users[0]}): ").strip()
  if not user_input:
    user_input = all_users[0]
  
  num_input = input("Number of recommendations (default 10): ").strip()
  num_recs = int(num_input) if num_input.isdigit() else 10
  
  display_recommendations(user_input, num_recs=num_recs)

Output()

## Section 7: Evaluate and Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

# Create a function to visualize recommendations
def visualize_recommendations(user_id, num_recs=10):
  """Create visualizations for user recommendations"""
  result = get_recommendations(user_id, num_recommendations=num_recs)
  
  if result['status'] == 'error':
    print(f"Error: {result['message']}")
    return
  
  recs_df = result['recommendations']
  
  # Create subplot layout
  fig, axes = plt.subplots(2, 2, figsize=(16, 12))
  fig.suptitle(f'Recommendation Analysis for {user_id}', fontsize=16, fontweight='bold')
  
  # Plot 1: Similarity Scores
  ax1 = axes[0, 0]
  bars = ax1.barh(range(len(recs_df)), recs_df['Similarity Score'], color='steelblue')
  ax1.set_yticks(range(len(recs_df)))
  ax1.set_yticklabels([t[:30] + '...' if len(t) > 30 else t for t in recs_df['Title']], fontsize=9)
  ax1.set_xlabel('Similarity Score', fontsize=11)
  ax1.set_title('Top Recommendations by Similarity Score', fontsize=12, fontweight='bold')
  ax1.invert_yaxis()
  
  # Add value labels on bars
  for i, (bar, score) in enumerate(zip(bars, recs_df['Similarity Score'])):
    ax1.text(score, i, f' {score:.3f}', va='center', fontsize=9)
  
  # Plot 2: Max vs Avg Similarity
  ax2 = axes[0, 1]
  x = range(len(recs_df))
  width = 0.35
  ax2.bar([i - width/2 for i in x], recs_df['Max Similarity'], width, label='Max Similarity', color='lightcoral')
  ax2.bar([i + width/2 for i in x], recs_df['Avg Similarity'], width, label='Avg Similarity', color='lightgreen')
  ax2.set_xticks(x)
  ax2.set_xticklabels(range(1, len(recs_df) + 1))
  ax2.set_ylabel('Similarity Score', fontsize=11)
  ax2.set_xlabel('Recommendation Rank', fontsize=11)
  ax2.set_title('Max vs Avg Similarity Scores', fontsize=12, fontweight='bold')
  ax2.legend()
  ax2.grid(axis='y', alpha=0.3)
  
  # Plot 3: Recommendation Rank vs Score
  ax3 = axes[1, 0]
  ax3.plot(recs_df['Rank'], recs_df['Similarity Score'], marker='o', linestyle='-', linewidth=2, markersize=8, color='darkgreen')
  ax3.fill_between(recs_df['Rank'], recs_df['Similarity Score'], alpha=0.3, color='green')
  ax3.set_xlabel('Recommendation Rank', fontsize=11)
  ax3.set_ylabel('Similarity Score', fontsize=11)
  ax3.set_title('Score Decay by Rank', fontsize=12, fontweight='bold')
  ax3.grid(True, alpha=0.3)
  
  # Plot 4: Genre Distribution of Recommendations
  ax4 = axes[1, 1]
  genre_counts = {}
  for genres_str in recs_df['Genres']:
    for genre in genres_str.split(','):
      genre = genre.strip()
      genre_counts[genre] = genre_counts.get(genre, 0) + 1
  
  if genre_counts:
    genres = list(genre_counts.keys())[:10] # Top 10 genres
    counts = [genre_counts[g] for g in genres]
    ax4.barh(genres, counts, color='purple', alpha=0.7)
    ax4.set_xlabel('Count', fontsize=11)
    ax4.set_title('Genre Distribution in Recommendations (Top 10)', fontsize=12, fontweight='bold')
    ax4.invert_yaxis()
  
  plt.tight_layout()
  plt.show()
  
  return fig


# INTERACTIVE VISUALIZATION - SAME USER AS ABOVE
# ===============================================
# Visualize recommendations for the selected user

try:
  from ipywidgets import Button, Output, HBox
  from IPython.display import display, clear_output
  
  # Create visualization button
  viz_button = Button(
    description='Show Visualization',
    button_style='success',
    tooltip='Click to generate visualization for selected user',
    icon='bar-chart'
  )
  
  viz_output = Output()
  
  # Define callback function
  def on_viz_button_click(b):
    with viz_output:
      clear_output(wait=True)
      selected_user = user_dropdown.value
      num_recs = num_recs_slider.value
      visualize_recommendations(selected_user, num_recs=num_recs)
  
  viz_button.on_click(on_viz_button_click)
  
  # Create new controls layout with both buttons
  buttons_layout = HBox([execute_button, viz_button])
  all_controls = VBox([
    user_dropdown,
    num_recs_slider,
    buttons_layout
  ])
  
  display(all_controls)
  display(output)
  display(viz_output)
  
except Exception as e:
  # Fallback: just show a sample visualization
  print(f"\nError setting up visualization: {e}\n")
  print("Showing sample visualization for User199:\n")
  visualize_recommendations('User199', num_recs=10)

Output()

Output()

## Summary: System Statistics and Performance

In [59]:
print("="*80)
print("CONTENT-BASED RECOMMENDATION SYSTEM - SUMMARY")
print("="*80)

print(f"""
Dataset Statistics:
 - Total Users:      {len(all_users)}
 - User ID Range:     {all_users[0]} to {all_users[-1]}
 - Total User-Book Pairs: {len(user_books_parsed)}
 - Unique Titles (Raw):  {user_books_parsed['Title'].nunique()}
 - Unique Titles (Classified): {len(books_unique)}
 - Classification Rate:  {(len(books_unique) / user_books_parsed['Title'].nunique() * 100):.1f}%

TF-IDF Vectorization:
 - Total Books in Catalog: {tfidf_matrix.shape[0]}
 - Feature Dimensions:   {tfidf_matrix.shape[1]}
 - Feature Space:     Title + Genres (emphasized 2x) + Language
 - Vectorizer:      TF-IDF with sublinear scaling
 - N-gram Range:     (1, 2) - unigrams and bigrams

Similarity Matrix:
 - Shape:         {similarity_matrix.shape}
 - Algorithm:       Cosine Similarity
 - Mean Similarity Score: {similarity_scores.mean():.4f}
 - Std Dev:        {similarity_scores.std():.4f}
 - Min Score:       {similarity_scores.min():.4f}
 - Max Score:       {similarity_scores.max():.4f}

Recommendation Algorithm:
 - Strategy:       Content-Based Filtering
 - Scoring:        60% Best Match + 40% Average Match
 - Item-Item Approach:  Based on genre and title similarity
 - Cold Start Handling:  Uses similarity matrix for new users

Genre Coverage:
 - Total Unique Genres:  {genres_df['Genres'].nunique()}
 - Languages Supported:  {genres_df['language'].nunique()}
 
Available Test Users:
 Recommended for testing: {', '.join(all_users[:10])}... and {len(all_users) - 10} more!
""")

print("\n" + "="*80)
print("HOW TO USE THIS SYSTEM:")
print("="*80)
print("""
1. Use the function: get_recommendations(user_id, num_recommendations=10)
  Example: get_recommendations('User199', num_recommendations=10)

2. Use the display function: display_recommendations(user_id, num_recs=10)
  Example: display_recommendations('User250', num_recs=15)

3. Use visualization: visualize_recommendations(user_id, num_recs=10)
  Example: visualize_recommendations('User180', num_recs=10)

4. Available user IDs: User002 through User299

QUICK START:
 → Use the interactive dropdown above to select any user
 → Click "Show Recommendations" to see text results
 → Click "Show Visualization" to see 4-subplot analysis
""")

CONTENT-BASED RECOMMENDATION SYSTEM - SUMMARY

Dataset Statistics:
 - Total Users:      121
 - User ID Range:     User002 to User299
 - Total User-Book Pairs: 1000
 - Unique Titles (Raw):  910
 - Unique Titles (Classified): 116
 - Classification Rate:  12.7%

TF-IDF Vectorization:
 - Total Books in Catalog: 116
 - Feature Dimensions:   500
 - Feature Space:     Title + Genres (emphasized 2x) + Language
 - Vectorizer:      TF-IDF with sublinear scaling
 - N-gram Range:     (1, 2) - unigrams and bigrams

Similarity Matrix:
 - Shape:         (116, 116)
 - Algorithm:       Cosine Similarity
 - Mean Similarity Score: 0.0942
 - Std Dev:        0.1523
 - Min Score:       0.0000
 - Max Score:       1.0000

Recommendation Algorithm:
 - Strategy:       Content-Based Filtering
 - Scoring:        60% Best Match + 40% Average Match
 - Item-Item Approach:  Based on genre and title similarity
 - Cold Start Handling:  Uses similarity matrix for new users

Genre Coverage:
 - Total Unique Genres:  111
 

## Advanced Usage & Customization

### How the Recommendation Algorithm Works

**Content-based filtering** recommends items (books) similar to items the user already likes.

**Process:**
1. **Feature Engineering**: Each book is represented by a "soup" containing:
  - Title words
  - Genre labels (given extra weight for importance)
  - Language information

2. **TF-IDF Vectorization**: Converts text into numerical vectors
  - TF (Term Frequency): How often a word appears in a book's soup
  - IDF (Inverse Document Frequency): How unique/rare that word is across all books
  - Result: Books with distinctive genres get higher TF-IDF scores

3. **Similarity Computation**: Cosine similarity measures how "close" two books are
  - Range: 0 (completely different) to 1 (identical)
  - Uses the angle between vectors (not Euclidean distance)

4. **Recommendation Scoring**:
  - For each unread book, compute similarity to ALL books user has read
  - Score = 60% (best match) + 40% (average match)
  - This balances "finding one very similar book" vs "finding consistently similar books"
  - Rank by score and return top N

### Why This Approach Is Effective

* **No cold-start problem** for users (works with any user instantly)
* **Genre-aware** (genres are emphasized in the feature soup)
* **Language-aware** (handles both English and Armenian books)
* **Scalable** (adds new users/books without retraining)
* **Interpretable** (easy to explain "why" a book is recommended)
* **Fast** (cosine similarity is computationally efficient)